# RAG Is Simpler Than You Think

## Build multimodal RAG one choice at a time

People ask questions in text, and the evidence lives in photos. You retrieve the right images, then a vision model answers from those photos.

Photos stay in one catalog. Try all six recipes, or stop when the results are good enough and the system is something your team can run.

- Search titles that come with each photo
- Caption each photo and search that text
- Search the image when the question uses different words
- Compare two CLIP models on the same question
- Rewrite a vague question into a short search query
- Answer from the retrieved photos with a vision model

Pixeltable applies each recipe to that catalog. You can add the next method when a question needs it.

This tutorial was inspired by Rafael Pierre's [RAG Is Simpler Than You Think](https://www.lighthousenewsletter.com/p/rag-is-simpler-than-you-think) and Hugo Bowne-Anderson's account of Doug Turnbull's talk, [RAG Isn't Dead. It's Sleeping.](https://www.vanishinggradients.com/p/rag-isnt-dead-its-sleeping)

## How we will compare the choices

Search `zebra` first. That word appears in two titles. Then search `a striped animal`, which describes the same photos with different words.

## How we will compare the choices

Search `zebra` first. That word appears in two titles. Then search `a striped animal`, which describes the same photos with different words.

Recipe 1 uses a photo and its title. Later recipes add a caption, image embeddings, a rewritten query, or an answer from the retrieved photos. Each recipe runs a question that shows what that method can do.

Each recipe has a Python file that describes the table it needs. `pxt schema diff FILE TARGET` shows what the file would add or change. `pxt schema update FILE TARGET` makes that change.

Run the notebook from this folder so those commands can find the files.

Each recipe has a Python file that describes the table it needs. `pxt schema diff FILE TARGET` shows what the file would add or change. `pxt schema update FILE TARGET` makes that change.

Run the notebook from this folder so those commands can find the files.

In [1]:
!uv sync --package simpler-rag

Resolved 189 packages in 15ms
Audited 160 packages in 40ms


In [2]:
!uv run pxt schema example

"""Pixeltable schema, written by 'pxt schema example'.

Every construct the schema DSL supports appears below; delete what you do not need.
'pxt schema example --brief' prints a minimal starting point instead.

'pxt schema diff FILE TARGET' reports what applying this would change; 'pxt schema update' applies it.

A udf defined here is referenced by this file's path, so moving or renaming the file leaves the columns that
call it unable to compute.
Building an application with Pixeltable? The agent skill carries the full API:
    npx skills add pixeltable/pixeltable-skill
"""

import pixeltable as pxt
import pixeltable.functions as pxtf

TableModel = pxt.model_base()


# a udf: a Python function the computed columns below can call
@pxt.udf
def excerpt(text: str, n: int = 80) -> str:
    return text if len(text) <= n else f'{text[:n]}...'


class Docs(TableModel, name='docs'):
    """One model becomes one table, named by name=."""

    # an annotation declares a stored column
    doc_id: 

In [3]:
!uv run pxt service example

"""Pixeltable application, written by 'pxt service example'.

One file holds both: the models, which name the tables, and the services, which serve routes over them.
The target given on the command line says which catalog directory those tables live in, so the same file
can be applied to a development directory and a production one.

    pxt schema update app.py TARGET        # create the tables the models declare
    pxt service update app.py TARGET       # serve this file's services against them

A udf defined here is referenced by this file's path, so moving or renaming the file leaves the columns that
call it unable to compute.
"""

from __future__ import annotations  # required to declare a model on Python 3.14+

import pixeltable as pxt
import pixeltable.functions as pxtf
from pixeltable.serving import FastAPIRouter

TableModel = pxt.model_base()


# a udf: a Python function the computed columns below can call
@pxt.udf
def excerpt(text: str, n: int = 12) -> str:
    return text i

## Recipe 1: search the image titles

Titles come with each photo. The schema file also declares `search_title`, a lowercase copy, so a search for `zebra` matches `Zebra grazing`. Keyword search uses the words already on the row.

In [ ]:
from pathlib import Path

print(Path('recipe1_store.py').read_text())

Preview the first migration. The diff should show one new table with the columns from the schema above. If that plan matches, apply it.

In [ ]:
!pxt schema diff recipe1_store.py rag_demo

In [ ]:
!pxt schema update recipe1_store.py rag_demo -f

Insert a small set of photos. Each row has a photo and a short title.

In [ ]:
import pixeltable as pxt

BASE = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/docs/resources/images/'

photos = pxt.get_table('rag_demo.photos')
photos.insert(
    [
        {
            'photo_id': 1,
            'image': BASE + '000000000001.jpg',
            'title': 'Taxi on a city street',
        },
        {
            'photo_id': 2,
            'image': BASE + '000000000009.jpg',
            'title': 'Bento lunch box',
        },
        {
            'photo_id': 3,
            'image': BASE + '000000000016.jpg',
            'title': 'Baseball batter',
        },
        {
            'photo_id': 4,
            'image': BASE + '000000000025.jpg',
            'title': 'Giraffe at the zoo',
        },
        {
            'photo_id': 5,
            'image': BASE + '000000000030.jpg',
            'title': 'Vase of flowers',
        },
        {
            'photo_id': 6,
            'image': BASE + '000000000034.jpg',
            'title': 'Zebra grazing',
        },
        {
            'photo_id': 7,
            'image': BASE + '000000000042.jpg',
            'title': 'Dog asleep on the shoes',
        },
        {
            'photo_id': 8,
            'image': BASE + '000000000049.jpg',
            'title': 'Horse riders performing',
        },
        {
            'photo_id': 9,
            'image': BASE + '000000000057.jpg',
            'title': 'Tennis match',
        },
        {
            'photo_id': 10,
            'image': BASE + '000000000061.jpg',
            'title': 'Elephant trek in the jungle',
        },
        {
            'photo_id': 11,
            'image': BASE + '000000000064.jpg',
            'title': 'Street clock',
        },
        {
            'photo_id': 12,
            'image': BASE + '000000000071.jpg',
            'title': 'Commuter train in the yard',
        },
        {
            'photo_id': 13,
            'image': BASE + '000000000080.jpg',
            'title': 'Zebra in an enclosure',
        },
        {
            'photo_id': 14,
            'image': BASE + '000000000090.jpg',
            'title': 'Cow in a pasture',
        },
    ]
)


In [ ]:
photos.select(
    photos.photo_id, photos.image, photos.title, photos.search_title
).head(3)

### Search the words we have

Keyword search finds a photo when the question reuses a word from the title. Search `search_title` so capitalization does not change the result.

In [ ]:
photos.where(
    photos.search_title.contains('zebra')
).select(
    photos.photo_id, photos.image, photos.title, photos.search_title
).collect()

Keyword search matches words that are already on the row. Ask for "a striped animal" and you get no rows. Two photos fit the question, but neither title uses those words.

In [ ]:
photos.where(photos.search_title.contains('striped')).select(
    photos.photo_id, photos.image, photos.title, photos.search_title
).collect()

## Recipe 2: turn images into searchable text

Titles omit most of what each photo shows. A captioning model writes a text description for every photo, which gives `contains()` more words to match. Those captions are still text.

In [ ]:
print(Path('recipe2_caption.py').read_text())

The `caption` column is new. Apply the schema, and the model writes captions for the photos already in the table. The first run downloads the model, so it can take a minute.

In [ ]:
!pxt schema diff recipe2_caption.py rag_demo

In [ ]:
!pxt schema update recipe2_caption.py rag_demo -f

In [ ]:
# Reload the table so Python sees the new caption column.
photos = pxt.get_table('rag_demo.photos')
photos.select(
    photos.photo_id, photos.image, photos.title, photos.caption
).collect()

Captions add more text, and they still describe each photo in a few words. The zebra captions use `zebra`, not `striped`, so this search returns no rows. The next recipe searches the images themselves.

In [ ]:
photos.where(photos.caption.contains('striped')).select(
    photos.photo_id, photos.image, photos.title, photos.caption
).collect()

## Recipe 3: search the image content directly

Captions reduce each image to one short description. CLIP compares a text question with the image itself, and each photo is already one row.

In [ ]:
print(Path('recipe3_search.py').read_text())

Apply the CLIP index to embed the photos already in the table. The first run downloads the CLIP model.

In [ ]:
!pxt schema diff recipe3_search.py rag_demo

In [ ]:
!pxt schema update recipe3_search.py rag_demo -f

CLIP finds the photos that keyword search missed by comparing a text query with what each photo shows. Sort by that score to put the closest matches first.

In [ ]:
# Reload the table so Python sees the new embedding index.
photos = pxt.get_table('rag_demo.photos')
sim = photos.image.similarity(string='a striped animal', idx='clip_idx')
photos.order_by(sim, asc=False).limit(3).select(
    photos.photo_id,
    photos.image,
    photos.title,
    photos.caption,
    score=sim,
).collect()

In [ ]:
sim = photos.image.similarity(
    string='something you can eat', idx='clip_idx'
)
photos.order_by(sim, asc=False).limit(3).select(
    photos.photo_id,
    photos.image,
    photos.title,
    photos.caption,
    score=sim,
).collect()

### Rerank the shortlist

Embedding search returns six candidates. The cross-encoder then compares the question with each caption and scores those six rows.

After `collect()` returns the six rows to Python, pandas sorts them by that new score. Pixeltable does the retrieval and scoring.

In [ ]:
import pixeltable.functions as pxtf

query = 'animals you would see on a safari'
model_id = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

sim = photos.image.similarity(string=query, idx='clip_idx')
shortlist = (
    photos.order_by(sim, asc=False)
    .limit(6)
    .select(
        photos.photo_id,
        photos.image,
        photos.title,
        photos.caption,
        similarity_score=sim,
        rerank_score=pxtf.huggingface.cross_encoder(query, photos.caption, model_id=model_id),
    )
    .collect()
)
display(shortlist)

shortlist.to_pandas().sort_values('rerank_score', ascending=False)[
    [
        'photo_id',
        'image',
        'title',
        'caption',
        'similarity_score',
        'rerank_score',
    ]
]

## Recipe 4: compare image embedding models

Different image embedding models rank the same photos differently. Run the same text question against both models and compare the results before you keep one.

In [ ]:
print(Path('recipe4_migrate.py').read_text())

Apply a second embedding index. The first index stays, and the new model embeds the existing photos.

In [ ]:
!pxt schema diff recipe4_migrate.py rag_demo

In [ ]:
!pxt schema update recipe4_migrate.py rag_demo -f

Run the same query on `clip_idx` and `clip_idx_v2`. Compare the rankings, then keep the model that ranks better.

In [ ]:
# Reload the table so Python sees both embedding indexes.
photos = pxt.get_table('rag_demo.photos')
query = 'public transportation'

clip32_score = photos.image.similarity(string=query, idx='clip_idx')
clip32_results = (
    photos.order_by(clip32_score, asc=False)
    .limit(2)
    .select(
        photos.photo_id,
        photos.image,
        photos.title,
        photos.caption,
        score=clip32_score,
    )
    .collect()
)
print('CLIP ViT-B/32')
display(clip32_results)

clip16_score = photos.image.similarity(string=query, idx='clip_idx_v2')
clip16_results = (
    photos.order_by(clip16_score, asc=False)
    .limit(2)
    .select(
        photos.photo_id,
        photos.image,
        photos.title,
        photos.caption,
        score=clip16_score,
    )
    .collect()
)
print('CLIP ViT-B/16')
display(clip16_results)

### Keep new photos searchable

New photos need a caption and an embedding from each model. Insert them, and Pixeltable fills those values so the same searches can find the new row.

In [ ]:
photos.insert(
    [
        {
            'photo_id': 15,
            'image': BASE + '000000000036.jpg',
            'title': 'New photo A',
        },
        {
            'photo_id': 16,
            'image': BASE + '000000000063.jpg',
            'title': 'New photo B',
        },
    ]
)

sim = photos.image.similarity(string='a close-up photo', idx='clip_idx')
photos.where(photos.photo_id >= 15).order_by(sim, asc=False).select(
    photos.photo_id,
    photos.image,
    photos.title,
    photos.caption,
    score=sim,
).collect()

## Recipe 5: understand the question before you search

People often ask vague questions, or they leave out the words that appear in the data. A model turns the request into a short description that keyword search or CLIP can use.

This notebook tests rewriting after the retrieval recipes, so those methods can share the same question. An application would rewrite first, then retrieve.

OpenAI runs the rewrite. Set `OPENAI_API_KEY` where the `pxt` daemon runs before you run the query cell.

In [ ]:
print(Path("recipe5_rewrite.py").read_text())

In [ ]:
!pxt schema update recipe5_rewrite.py rag_demo -f

In [ ]:
queries = pxt.get_table('rag_demo.queries')
question = 'what are the big animals with really long necks?'
rewrite_status = queries.insert([{'raw': question}], return_rows=True)
rewritten_row = rewrite_status.rows[0]

print(f"Raw question: {rewritten_row['raw']}")
print(f"Search query: {rewritten_row['search_query']}")

clean = rewritten_row['search_query']
sim = photos.image.similarity(string=clean, idx='clip_idx')
retrieved = (
    photos.order_by(sim, asc=False)
    .limit(3)
    .select(
        photos.photo_id,
        photos.image,
        photos.title,
        photos.caption,
        score=sim,
    )
    .collect()
)
display(retrieved)

## Recipe 6: answer from the retrieved images

Retrieval returns a set of photos. A vision model then uses those photos as evidence for an answer.

Give the model the three retrieved images, plus each photo's ID, title, and caption. The prompt tells the model to answer only from that context and cite the photo IDs it used.

In [ ]:
print(Path('recipe6_answer.py').read_text())

Each row in `answers` keeps the question, the three retrieved images, notes about their source photo IDs, and the answer from the vision model.

In [ ]:
!pxt schema diff recipe6_answer.py rag_demo
!pxt schema update recipe6_answer.py rag_demo -f

In [ ]:
source_notes = '\n'.join(
    f"Photo {retrieved['photo_id'][i]}: {retrieved['title'][i]}. "
    f"Caption: {retrieved['caption'][i]}"
    for i in range(3)
)

answers = pxt.get_table('rag_demo.answers')
answer_status = answers.insert(
    [
        {
            'question': question,
            'source_notes': source_notes,
            'image_1': retrieved['image'][0],
            'image_2': retrieved['image'][1],
            'image_3': retrieved['image'][2],
        }
    ],
    return_rows=True,
)

print('Answer:')
print(answer_status.rows[0]['answer'])
print('\nRetrieved evidence:')
display(retrieved)

## Choose what your team can support

Match the recipe to the questions people ask and the work your team can support. Keyword search works when questions reuse the words in your titles. Query rewriting helps when people ask in a roundabout way. Image embeddings and reranking cover questions that still miss.

The same `photos` table can hold titles, captions, indexes, and answers from retrieved photos. Add the next recipe when the current one fails a question you care about.

To start over, drop the directory and run the notebook again:

In [ ]:
# pxt.drop_dir('rag_demo', force=True)